In [ ]:
from datasets import Dataset, load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer, DataCollatorForLanguageModeling
from itertools import chain
import pandas as pd
from pathlib import Path

In [ ]:
# Our data is located in the /data directory. Let's see what we're working with.
#!ls ./data # !ls is linux compatible command to list files in a directory. You can also use the file explorer in vscode to see the files in the data directory.
#!dir .\data # dir is the windows compatible command to list files in a directory. You can also use the file explorer in vscode to see the files in the data directory.

In [ ]:
# Load the file names into a list
data_path = Path("./data")
file_paths = [filename for filename in data_path.glob("*.txt")]

In [ ]:
# Read all the file contents into a list
file_data = list()
for filename in file_paths:
    with open(filename, "r", encoding="utf-8") as f:
        data = f.read()
    file_data.append(data)

In [ ]:
print(file_data[0])
print('=====================')
print(file_data[1])

In [ ]:
# Convert our list of text into a dataset using .from_dict()
dataset = Dataset.from_dict({"text": file_data})

In [ ]:
# Preview the dataset
dataset["text"]

In [ ]:
# Load the tokenizer for GPT-2
tokenizer = AutoTokenizer.from_pretrained('gpt2')

# The tokenizer does not have a pad token, so we'll specify one.
tokenizer.pad_token = tokenizer.eos_token

# Load the GPT-2 model
model = AutoModelForCausalLM.from_pretrained('gpt2')

In [ ]:
# Create a tokenization function to tokenize the dataset
def tokenize_function(examples):
    output = tokenizer(examples['text'])
    return output

# Run the tokenizer over our dataset using the .map method 
# NOTE: For large datasets, this can take a while
tokenized_dataset = dataset.map(tokenize_function, batched=True)

# We want to remove our original dataset's column names from the tokenized dataset
tokenized_dataset = tokenized_dataset.remove_columns(dataset.column_names)

In [ ]:
tokenized_dataset

In [ ]:
# This function was lightly modified from the HuggingFace run_clm.py
# You can find the original function at https://github.com/huggingface/transformers/blob/main/examples/pytorch/language-modeling/run_clm.py
# Create a preprocessing function to group aour texts together in chunks of 1024
def group_texts(examples):
    # Specify our bock size -- 1024
    block_size = 1024
    
    # Concatenate all the texts together for each example
    concatenated_examples = dict()
    for k in examples.keys():
        concatenated_examples[k] = list(chain(*examples[k]))
        
    # Compute the total length of all the text
    total_length = len(concatenated_examples[list(examples.keys())[0]])
    
    # We drop the small remainder of the block
    # If total_length < block_size, we return an empty dict.
    total_length = (total_length // block_size) * block_size
    
    # Split into chunks of 1024
    result = dict()
    # Loop over the keys and texts in the concatenated examples
    for k, t in concatenated_examples.items():
        # Divide each text into chunks of 1024
        chunks = list()
        for i in range(0, total_length, block_size):
            chunks.append(t[i : i + block_size])
        result[k] = chunks
    # Set the "labels" equal to the "input_ids"
    result["labels"] = result["input_ids"].copy()
    return result

In [ ]:
# Chunk our datasets using the group_texts function
dataset = tokenized_dataset.map(group_texts, batched=True)

In [ ]:
# Set up our data collator for training. Since our model is PyTorch, we need to specify return_tensors as "pt"
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False, return_tensors="pt")

In [ ]:
# Establish our training arguments
training_args = TrainingArguments(
    output_dir="finetune_gpt2",
    per_device_train_batch_size=1,
    save_strategy="no"
)

In [ ]:
# Put everything into our Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    data_collator=data_collator
)

In [ ]:
# Run the trainer
trainer.train()

In [ ]:
# Specify an input string
input_string = "Cross-Site Scripting is a vulnerability that"

# Tokenize our input string
input_ids = tokenizer(input_string, return_tensors="pt").input_ids

# Generate model output_ids
outputs = model.generate(
    input_ids,
    num_beams=10,
    num_return_sequences=1,
    no_repeat_ngram_size=1,
    remove_invalid_values=True,
)

# Decode the output tokens to text
output_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

# Print our output!
print(output_text)